# Forecast Seviri all Bands with OpenCV
Conda Env: scikit-learn
created: 02.07.2021

In [ ]:
import numpy as np
import pandas as pd
import cv2
import os
from osgeo import gdal
from datetime import datetime, timedelta
import datetime
import shlex

### set variables

In [ ]:
date_predict = datetime.datetime.strptime("29.01.2018 07:35:11", "%d.%m.%Y %H:%M:%S")

eumetsat_path = "C:\\Users\\Andreas\\Documents\\UNIGIS\\2017\\Master-Thesis\\Daten\\Satellite\\EUMETSAT"
eumetsat_geotiff_path = eumetsat_path + "\\Result_Timestamped\\GeoTIFF\\Extended_Clip"
eumetsat_forecast_output_path = eumetsat_path + "\\Forecast"

forecast_range = 180
forecast_step = 15

### set date and time for last and next to last Image

In [ ]:
date_last_image_rounded = date_predict - (date_predict - date_predict.min) % timedelta(minutes=15)
date_last_image=date_last_image_rounded.strftime("%Y-%m-%d %H_%M_%S")
date_next_to_last_rounded = date_last_image_rounded - datetime.timedelta(minutes=15)
date_next_to_last_image=date_next_to_last_rounded.strftime("%Y-%m-%d %H_%M_%S")

### load all images and bands from geotiff into numpy arrays

In [ ]:
images=["next_to_last_image","last_image"]
for image in images:
    if image == "next_to_last_image":
        date_processing = date_next_to_last_image
    else:
        date_processing = date_last_image
    datasets = ["IR_VIS_WR","HRV"]
    for dataset in datasets:
        if dataset == "HRV":
            bands = ["HRV"]
        else:
            bands = ['VIS006','VIS008','IR_016','IR_039','WV_062','WV_073','IR_087','IR_097','IR_108','IR_120','IR_134']
        band_nr = 1
        for band in bands:
            geotiff_filename = os.path.join(eumetsat_geotiff_path, date_processing + "_{}.tif".format(dataset))
            image_processing = gdal.Open(geotiff_filename)
            dynamic_array = np.array(image_processing.GetRasterBand(band_nr).ReadAsArray().astype(np.float32))
            globals()[image + "_"+ band] = dynamic_array
            band_nr = band_nr + 1
data_type = image_processing.GetRasterBand(1).DataType
geotransform = image_processing.GetGeoTransform()
spatialreference = image_processing.GetProjection()
ncol = image_processing.RasterXSize
nrow = image_processing.RasterYSize
nband = 1

### Detect motion flow from HRV-Files and predict on all bands

In [ ]:
def export_geotiff(path, file, band, ncol, nrow, nband, data_type, geotransform, spatialreference, image):
    if not os.path.exists(path + "\\" + band):
        os.makedirs(path + "\\" + band)
    output_geotiff_file = os.path.join(path + "\\" + band, file + ".tif")
    driver = gdal.GetDriverByName("GTiff")
    dst_dataset = driver.Create(output_geotiff_file, ncol, nrow, nband, data_type, ['COMPRESS=PACKBITS','TILED=YES','NUM_THREADS=ALL_CPUS'])
    dst_dataset.SetGeoTransform(geotransform)
    dst_dataset.SetProjection(spatialreference)
    dst_dataset.GetRasterBand(1).SetDescription(band)
    dst_dataset.GetRasterBand(1).WriteArray(image)
    dst_dataset = None

def print_array(name, array):
    print("  --{} min: {} max: {}".format(name, str(np.min(array)), str(np.max(array))))

def array2raster2(path, fname, matriz, geot, proj):
    if not os.path.exists(path):
        os.makedirs(path)
    drv = gdal.GetDriverByName("GTiff")
    dst_ds = drv.Create(os.path.join(path + "\\", fname), matriz.shape[1], matriz.shape[0], 3, gdal.GDT_Byte)
    dst_ds.SetGeoTransform(geot)
    dst_ds.SetProjection(proj)
    dst_ds.GetRasterBand(3).WriteArray(matriz[:,:,0])
    dst_ds.GetRasterBand(2).WriteArray(matriz[:,:,1])
    dst_ds.GetRasterBand(1).WriteArray(matriz[:,:,2])
    dst_ds.FlushCache()
    dst_ds=None

def draw_flow(img, flow, step=16):
    h, w = img.shape[:2]
    y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)
    fx, fy = flow[y,x].T*5
    lines = np.vstack([x, y, x+fx, y+fy]).T.reshape(-1, 2, 2)
    lines = np.int32(lines + 0.5)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for (x1, y1), (_x2, _y2) in lines:
        cv2.arrowedLine(vis, (x1, y1), (_x2, _y2), (0, 255, 0), 1, 1, 0, 0.5)
    return vis

In [ ]:
all_bands = ['HRV','VIS006','VIS008','IR_016','IR_039','WV_062','WV_073','IR_087','IR_097','IR_108','IR_120','IR_134']

for band in all_bands:
    next_to_last = eval(images[0] + "_" + band)
    last = eval(images[1] + "_" + band)
    concatenated = np.concatenate((next_to_last, last), axis=0)
    globals()[images[0]+"_"+band+"_scaled"] = np.uint8(next_to_last / np.max(concatenated) * 255)
    globals()[images[1]+"_"+band+"_scaled"] = np.uint8(last / np.max(concatenated) * 255)
    globals()[band+"_concatenated"] = concatenated
    export_geotiff(eumetsat_forecast_output_path, date_last_image+" -15min_"+band, band, ncol, nrow, nband, data_type, geotransform, spatialreference, next_to_last)
    export_geotiff(eumetsat_forecast_output_path, date_last_image+" +0min_"+band, band, ncol, nrow, nband, data_type, geotransform, spatialreference, last)

In [ ]:
hsv = np.zeros([nrow,ncol,3], dtype=np.uint8)
hsv[..., 1] = 255
counter = forecast_step

while counter <= forecast_range:
    forecast_name = date_last_image + " +" + str(counter) + "min"
    optical_flow = cv2.optflow.DualTVL1OpticalFlow_create(0.1, 0.003, 0.3, 6, 6, 0.005, 30, 2, 0.5, 0.2, 5, 0)
    flow = optical_flow.calc(eval(images[0]+"_HRV_scaled"), eval(images[1]+"_HRV_scaled"), None)
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    colored_flow = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    h, w = flow.shape[:2]
    flow_transformed = -flow
    flow_transformed[:,:,0] += np.arange(w)
    flow_transformed[:,:,1] += np.arange(h)[:,np.newaxis]
    for band in all_bands:
        last_arr = eval(images[1]+"_"+band)
        last_scaled = eval(images[1]+"_"+band+"_scaled")
        concat = eval(band+"_concatenated")
        forecast_scaled = cv2.remap(last_scaled, flow_transformed, None, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(255,255,255,255))
        if band == "HRV":
            array2raster2(eumetsat_forecast_output_path+"\\FLOW", forecast_name+"_FLOW.tif", colored_flow, geotransform, spatialreference)
            array2raster2(eumetsat_forecast_output_path+"\\HRV_ARROWS", forecast_name+"_HRV_ARROWS.tif", draw_flow(forecast_scaled, flow, 25), geotransform, spatialreference)
        forecast_arr = (forecast_scaled/255 * np.max(concat)).astype(float)
        export_geotiff(eumetsat_forecast_output_path, forecast_name+"_"+band, band, ncol, nrow, nband, data_type, geotransform, spatialreference, forecast_arr)
        globals()[images[0]+"_"+band+"_scaled"] = last_scaled.copy()
        globals()[images[1]+"_"+band+"_scaled"] = forecast_scaled.copy()
        globals()[images[0]+"_"+band] = last_arr.copy()
        globals()[images[1]+"_"+band] = forecast_arr.copy()
    counter += forecast_step
    cv2.waitKey(0)
cv2.destroyAllWindows()